# Implementing an LLM-powered recommendation system

## 任务介绍
### 根据用户描述的需求，在电影池中，向用户推荐合适的电影。
### 1.数据预处理
### 2.电影内容 embedding
### 3.构造检索模块
### 4. 基于用户信息和query，向用户推荐合适的电影

In [39]:
### 1.数据预处理

In [1]:
# 数据读取
import pandas as pd

anime = pd.read_csv('anime_with_synopsis.csv')
anime.head()
## 数据为电影分类及描述数据
## Name 电影名称
## Score 电影评分
## Genres 电影分类标签
## sypnipsis 电影内容描述

,MAL_ID,Name,Score,Genres,sypnopsis
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space","In the year 2071, humanity has colonized sever..."
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space","other day, another bounty—such is the life of ..."
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen","Vash the Stampede is the man with a $$60,000,0..."
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",ches are individuals with special powers like ...
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",It is the dark century and the people are suff...


In [2]:
anime = anime.dropna()

In [3]:

anime['combined_info'] = anime.apply(lambda row: f"Title: {row['Name']}. Overview: {row['sypnopsis']} Genres: {row['Genres']}", axis=1)
anime['combined_info'][0]



'Title: Cowboy Bebop. Overview: In the year 2071, humanity has colonized several of the planets and moons of the solar system leaving the now uninhabitable surface of planet Earth behind. The Inter Solar System Police attempts to keep peace in the galaxy, aided in part by outlaw bounty hunters, referred to as "Cowboys." The ragtag team aboard the spaceship Bebop are two such individuals. Mellow and carefree Spike Spiegel is balanced by his boisterous, pragmatic partner Jet Black as the pair makes a living chasing bounties and collecting rewards. Thrown off course by the addition of new members that they meet in their travels—Ein, a genetically engineered, highly intelligent Welsh Corgi; femme fatale Faye Valentine, an enigmatic trickster with memory loss; and the strange computer whiz kid Edward Wong—the crew embarks on thrilling adventures that unravel each member\'s dark and mysterious past little by little. Well-balanced with high density action and light-hearted comedy, Cowboy Bebo

In [4]:
len(anime)

16206

## 电影内容 Embeddings

In [5]:
# 1. 过滤掉过长的内容
import tiktoken
# embedding model parameters
embedding_encoding = "cl100k_base"  # this the encoding for text-embedding-ada-002
max_tokens = 8000  # the maximum for text-embedding-ada-002 is 8191

encoding = tiktoken.get_encoding(embedding_encoding)

# omit reviews that are too long to embed
anime["n_tokens"] = anime.combined_info.apply(lambda x: len(encoding.encode(x)))
anime = anime[anime.n_tokens <= max_tokens]
len(anime)

16206

In [6]:
anime.head()

,MAL_ID,Name,Score,Genres,sypnopsis,combined_info,n_tokens
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space","In the year 2071, humanity has colonized sever...",Title: Cowboy Bebop. Overview: In the year 207...,245
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space","other day, another bounty—such is the life of ...",Title: Cowboy Bebop: Tengoku no Tobira. Overvi...,199
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen","Vash the Stampede is the man with a $$60,000,0...",Title: Trigun. Overview: Vash the Stampede is ...,252
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",ches are individuals with special powers like ...,Title: Witch Hunter Robin. Overview: ches are ...,125
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",It is the dark century and the people are suff...,Title: Bouken Ou Beet. Overview: It is the dar...,188


In [8]:
!pip install modelscope

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 26.5 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [9]:
# embedding 模型文件下载，大陆地区建议使用modelscope
#SJ007NB/text2vec-base-multilingual
from modelscope import snapshot_download
# model_dir = snapshot_download('Jerry0/text2vec-base-chinese',cache_dir="./")
model_dir = snapshot_download('SJ007NB/text2vec-base-multilingual',cache_dir="./")

2025-06-29 18:45:58,597 - modelscope - INFO - Got 13 files, start to download ...


Processing 13 items:   0%|          | 0.00/13.0 [00:00<?, ?it/s]

2025-06-29 18:47:36,920 - modelscope - INFO - Download model 'SJ007NB/text2vec-base-multilingual' successfully.


In [14]:
!pip install langchain_community
!pip install -U text2vec

  Using cached numpy-2.0.2-cp39-cp39-macosx_10_9_x86_64.whl.metadata (60 kB)
Using cached numpy-2.0.2-cp39-cp39-macosx_10_9_x86_64.whl (21.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.0
    Uninstalling numpy-1.23.0:
      Successfully uninstalled numpy-1.23.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.6.0 requires daal==2021.4.0, which is not installed.
node2vec 0.4.6 requires numpy<2.0.0,>=1.19.5, but you have numpy 2.0.2 which is incompatible.
numba 0.55.1 requires numpy<1.22,>=1.18, but you have numpy 2.0.2 which is incompatible.
scipy 1.9.1 requires numpy<1.25.0,>=1.18.5, but you have numpy 2.0.2 which is incompatible.
tensorflow 2.13.0 requires numpy<=1.24.3,>=1.22, but you have numpy 2.0.2 which is incompatible.
tensorflow 2.13.0 requires typing-extensions<4.6.0,>=3.6.6, but you have typing-extensions 


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 2.2 MB/s eta 0:00:0000:0100:02
  Created wheel for text2vec: filename=text2vec-1.3.7-py3-none-any.whl size=73628 sha256=8686f8edd2537cc98dac3468309fba6ce280f218bad56fca3d946f1d71bb5a0c
  Stored in directory: /Users/lhc456/Library/Caches/pip/wheels/77/90/7b/0195a62be9bc8a2b30d2365a80ecce5f575ccf12d80316bac2
Successfully built text2vec
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the foll

In [ ]:
# 利用嵌入模型进行文本表示

from langchain_community.embeddings.text2vec import Text2vecEmbeddings

embedding = Text2vecEmbeddings(model_name_or_path = r"SJ007NB/text2vec-base-multilingual")
embedding.embed_documents([
    "This is a CoSENT(Cosine Sentence) model.",
    "It maps sentences to a 768 dimensional dense vector space.",
])
embedding.embed_query(
    "It can be used for text matching or semantic search."
)

In [ ]:
anime["embedding"] = anime.combined_info.apply(lambda x: embedding.embed_query(x))
anime.head()

In [ ]:
anime.rename(columns = {'embedding': 'vector'}, inplace = True)
anime.rename(columns = {'combined_info': 'text'}, inplace = True)
anime.to_pickle('anime2.pkl')

## 3. 构建检索召回能力

In [1]:
import pandas as pd

anime = pd.read_pickle('anime2.pkl')

anime.head(2)

,MAL_ID,Name,Score,Genres,sypnopsis,text,n_tokens,vector
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space","In the year 2071, humanity has colonized sever...",Title: Cowboy Bebop. Overview: In the year 207...,245,"[0.11087228, 0.2883531, -0.20313735, -0.032440..."
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space","other day, another bounty—such is the life of ...",Title: Cowboy Bebop: Tengoku no Tobira. Overvi...,199,"[0.09808712, 0.2053806, -0.15396793, -0.060929..."


In [2]:
anime['text'][0]

'Title: Cowboy Bebop. Overview: In the year 2071, humanity has colonized several of the planets and moons of the solar system leaving the now uninhabitable surface of planet Earth behind. The Inter Solar System Police attempts to keep peace in the galaxy, aided in part by outlaw bounty hunters, referred to as "Cowboys." The ragtag team aboard the spaceship Bebop are two such individuals. Mellow and carefree Spike Spiegel is balanced by his boisterous, pragmatic partner Jet Black as the pair makes a living chasing bounties and collecting rewards. Thrown off course by the addition of new members that they meet in their travels—Ein, a genetically engineered, highly intelligent Welsh Corgi; femme fatale Faye Valentine, an enigmatic trickster with memory loss; and the strange computer whiz kid Edward Wong—the crew embarks on thrilling adventures that unravel each member\'s dark and mysterious past little by little. Well-balanced with high density action and light-hearted comedy, Cowboy Bebo

In [3]:
anime["metadata"] = anime["text"].apply(lambda x:{"text":x} )

In [4]:
anime.head(2)

,MAL_ID,Name,Score,Genres,sypnopsis,text,n_tokens,vector,metadata
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space","In the year 2071, humanity has colonized sever...",Title: Cowboy Bebop. Overview: In the year 207...,245,"[0.11087228, 0.2883531, -0.20313735, -0.032440...",{'text': 'Title: Cowboy Bebop. Overview: In th...
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space","other day, another bounty—such is the life of ...",Title: Cowboy Bebop: Tengoku no Tobira. Overvi...,199,"[0.09808712, 0.2053806, -0.15396793, -0.060929...",{'text': 'Title: Cowboy Bebop: Tengoku no Tobi...


In [5]:
!pip install lancedb==0.21.0
# !pip install pyarrow==20.0.0
# !pip list | grep -i "lancedb"



[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [6]:
import lancedb
from lancedb.db import LanceDBConnection

uri = "dataset/sample-anime-lancedb2"

db_connection = LanceDBConnection(uri = uri)
db = lancedb.connect(uri)
table = db.create_table("anime2", anime, mode="overwrite")

[2025-06-29T11:22:03Z WARN  lance::dataset::write::insert] No existing dataset at /Users/lhc456/Downloads/01623 - AI人工智能算法工程师-sz/第30周 搜索与推荐：NLP在实际场景中的应用/3-实战：大模型推荐系统实战/附件/代码 2/dataset/sample-anime-lancedb2/anime2.lance, it will be created


In [7]:
table.schema

MAL_ID: int64
Name: string
Score: string
Genres: string
sypnopsis: string
text: string
n_tokens: int64
vector: fixed_size_list<item: float>[384]
  child 0, item: float
metadata: struct<text: string>
  child 0, text: string

In [ ]:
# from langchain.embeddings import OpenAIEmbeddings
from langchain_community.embeddings.text2vec import Text2vecEmbeddings
from langchain.vectorstores import LanceDB
from langchain.chains import RetrievalQA
import os

# embeddings = OpenAIEmbeddings(engine="text-embedding-ada-002", openai_api_key=openai.api_key)
embeddings = Text2vecEmbeddings(model_name_or_path = r"SJ007NB/text2vec-base-multilingual")

docsearch = LanceDB(connection = db_connection, embedding = embeddings, table_name="anime2")


In [ ]:
query = "I'm looking for an animated action movie. What could you suggest to me?"
docs = docsearch.similarity_search(query, k=3)
docs
# docs[0].page_content

In [ ]:
import os

os.environ["AZURE_OPENAI_API_KEY"] = ""
os.environ["AZURE_OPENAI_ENDPOINT"] = ""
os.environ["AZURE_OPENAI_API_VERSION"] = ""
os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"] = ""
from langchain_openai import AzureChatOpenAI

llmd = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    openai_api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)

qa = RetrievalQA.from_chain_type(llm=llmd, 
chain_type="stuff", retriever=docsearch.as_retriever(), return_source_documents=True)

query = "I'm looking for an action anime. What could you suggest to me? response me with Chinese"
result = qa({"query": query})
result['result']

In [ ]:
result

In [ ]:
# 从限定的数据里面推荐
df_filtered = anime[anime['Genres'].apply(lambda x: 'Action' in x)]
qa = RetrievalQA.from_chain_type(llm=llmd, chain_type="stuff", 
    retriever=docsearch.as_retriever(search_kwargs={'data': df_filtered}), return_source_documents=True)

query = "I'm looking for an anime with animals and an adventurous plot. response me with simple Chinese"
result = qa({"query": query})
result['result']

## Prompt engineering

In [96]:
from langchain.prompts import PromptTemplate

template = """You are a movie recommender system that help users to find anime that match their preferences. 
Use the following pieces of context to answer the question at the end. 
For each question, suggest three anime, with a short description of the plot and the reason why the user migth like it.
If you don't know the answer, just say that you don't know, don't try to make up an answer. please response with Chinese.

{context}

Question: {question}
Your response:"""


PROMPT = PromptTemplate(
    template=template, input_variables=["context", "question"])

chain_type_kwargs = {"prompt": PROMPT}


qa = RetrievalQA.from_chain_type(llm=llmd, 
    chain_type="stuff", 
    retriever=docsearch.as_retriever(),
    return_source_documents=True, 
    chain_type_kwargs=chain_type_kwargs)

query = "I'm looking for an action anime with animals, any suggestions?"
result = qa({'query':query})
print(result['result'])


1. 《兽耳娘动物园》（Kemono Friends）：这是一部发生在废弃动物园的故事，主人公是一位迷失方向的人类女孩和各种拥有动物特征的“兽耳娘”。他们一起探索园区，解开园区的秘密。之所以推荐这部动画，是因为它以动物为主题，结合了探险和轻微的动作元素。

2. 《野良神》（Noragami）：虽然主角不是动物，但这部动画中有一位重要角色是一只名叫雪音的神灵武器，可以变成一只猫。故事主要讲述了一个被遗忘的神“夜斗”和他的伙伴们的冒险。推荐这部动画是因为它有精彩的动作场面和神灵这一奇幻元素。

3. 《狼的孩子雨和雪》（Ookami Kodomo no Ame to Yuki）：这是一部电影，讲述了一位年轻母亲如何抚养她那半人半狼的两个孩子。虽然它更注重家庭和成长的主题，但是作为一部有着动物特征的角色的动画，它展示了人与自然之间的联系，并且包含了一些动作元素。推荐这部电影因为它有着深刻的情感和动人的故事。


In [97]:
result

{'query': "I'm looking for an action anime with animals, any suggestions?",
 'result': '1. 《兽耳娘动物园》（Kemono Friends）：这是一部发生在废弃动物园的故事，主人公是一位迷失方向的人类女孩和各种拥有动物特征的“兽耳娘”。他们一起探索园区，解开园区的秘密。之所以推荐这部动画，是因为它以动物为主题，结合了探险和轻微的动作元素。\n\n2. 《野良神》（Noragami）：虽然主角不是动物，但这部动画中有一位重要角色是一只名叫雪音的神灵武器，可以变成一只猫。故事主要讲述了一个被遗忘的神“夜斗”和他的伙伴们的冒险。推荐这部动画是因为它有精彩的动作场面和神灵这一奇幻元素。\n\n3. 《狼的孩子雨和雪》（Ookami Kodomo no Ame to Yuki）：这是一部电影，讲述了一位年轻母亲如何抚养她那半人半狼的两个孩子。虽然它更注重家庭和成长的主题，但是作为一部有着动物特征的角色的动画，它展示了人与自然之间的联系，并且包含了一些动作元素。推荐这部电影因为它有着深刻的情感和动人的故事。',
 'source_documents': [Document(metadata={'text': 'Title: Kämpfer Picture Drama. Overview: very special episode included with the Blu-ray release. The Entrails Animals and Akane discuss philosophical topics such as existence, purpose, and low-quality merchandise. Genres: Comedy'}, page_content='Title: Kämpfer Picture Drama. Overview: very special episode included with the Blu-ray release. The Entrails Animals and Akane discuss philosophical topics such as existence, purpose, and low-quality mer

In [98]:
from langchain.prompts import PromptTemplate

template_prefix = """You are a movie recommender system that help users to find anime that match their preferences. 
Use the following pieces of context to answer the question at the end. 
For each question, take into account the context and the personal information provided by the user.
If you don't know the answer, just say that you don't know, don't try to make up an answer. please response with Chinese.

{context}"""

user_info = """This is what we know about the user, and you can use this information to better tune your research:
Age: {age}
Gender: {gender}"""

template_suffix= """Question: {question}
Your response:"""

user_info = user_info.format(age = 38, gender = 'female')

COMBINED_PROMPT = template_prefix +'\n'+ user_info +'\n'+ template_suffix
print(COMBINED_PROMPT)


You are a movie recommender system that help users to find anime that match their preferences. 
Use the following pieces of context to answer the question at the end. 
For each question, take into account the context and the personal information provided by the user.
If you don't know the answer, just say that you don't know, don't try to make up an answer. please response with Chinese.

{context}
This is what we know about the user, and you can use this information to better tune your research:
Age: 38
Gender: female
Question: {question}
Your response:


In [99]:
PROMPT = PromptTemplate(
    template=COMBINED_PROMPT, input_variables=["context", "question"])

chain_type_kwargs = {"prompt": PROMPT}
qa = RetrievalQA.from_chain_type(llm=llmd, 
    chain_type="stuff", 
    retriever=docsearch.as_retriever(),
    return_source_documents=True, 
    chain_type_kwargs=chain_type_kwargs)

query = "I'm looking for an action anime with human, any suggestions?"
result = qa({'query':query})
print(result['result'])



您好！根据您的喜好，我推荐您观看《天才派对》（Genius Party）。这部动画是由七个独立且独特的短片组成的合集，包含了从感人到奇异的多种故事。每个故事都不同，您会遇到上学的怪物、一个难以处理自己问题的男人，以及一个通过艰难方式了解生命循环的孩子，还有更多独特的角色和体验。这部动画集合了日本最有创造力的艺术家的思维，真正是一个天才派对的场景。其中的动作、科幻元素可能会满足您对动作类动漫的喜好。希望您会喜欢这个推荐！


In [61]:
result['source_documents']

[Document(metadata={'text': 'Title: Kämpfer Picture Drama. Overview: very special episode included with the Blu-ray release. The Entrails Animals and Akane discuss philosophical topics such as existence, purpose, and low-quality merchandise. Genres: Comedy'}, page_content='Title: Kämpfer Picture Drama. Overview: very special episode included with the Blu-ray release. The Entrails Animals and Akane discuss philosophical topics such as existence, purpose, and low-quality merchandise. Genres: Comedy'),
 Document(metadata={'text': 'Title: Zetsumetsu Kigu-shun.. Overview: dialogue-less anime about endangered species where they only say "-shun.". Episodes are streamed on the anime\'s official Youtube channel and Twitter, and Kadokawa\'s official Youtube channel as they are co-creators of the brand. Genres: Kids'}, page_content='Title: Zetsumetsu Kigu-shun.. Overview: dialogue-less anime about endangered species where they only say "-shun.". Episodes are streamed on the anime\'s official Yout